# Setup:

In [ ]:
import torch
from transformers import AdamW, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, get_cosine_schedule_with_warmup
from tqdm import tqdm
import os
from datasets import Dataset
import random
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
import numpy as np
from collections import defaultdict
from tqdm import tqdm

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
DATASET_ROOT = "../../CrossVul"
ALLOWED_CWE_IDS = {"CWE-22"} # "CWE-22", "CWE-89", "CWE-787"
LANGUAGES = ['c', 'cpp', 'cs', 'java', 'py', 'php']
SEED = 42
EPOCHS = 3

In [2]:
vulBERTa = "claudios/VulBERTa-MLP-ReVeal"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(vulBERTa, trust_remote_code=True)
print(device)

cuda


In [3]:
class FileAwareTrainer(Trainer):
    def __init__(self, *args, eval_dataset_filenames=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.eval_dataset_filenames = eval_dataset_filenames

    def evaluate(self, eval_dataset=None, **kwargs):
        output = super().evaluate(eval_dataset=eval_dataset, **kwargs)
        self._last_eval_preds = kwargs.get('preds', None)
        return output

    def predict(self, test_dataset, **kwargs):
        self.eval_dataset_filenames = test_dataset['filename']
        return super().predict(test_dataset, **kwargs)

# Data Preprocessing

In [4]:
def collect_files_for_cwe(cwe_id):
    samples = []
    for lang in LANGUAGES:
        lang_dir = os.path.join(DATASET_ROOT, cwe_id, lang)
        if not os.path.isdir(lang_dir):
            continue
        for filename in os.listdir(lang_dir):
            filepath = os.path.join(lang_dir, filename)
            if filename.endswith('.DS_Store'):
                continue
            label = 1 if "bad" in filename.lower() else 0
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                code = f.read()
            samples.append({
                "filename": filename,
                "code": code,
                "label": label
            })
    print(len(samples))
    return samples

def compute_file_metrics_builder(filenames):
    def compute_file_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        file_pred_chunks = defaultdict(list)
        file_label = {}

        for pred, label, fname in zip(preds, labels, filenames):
            file_pred_chunks[fname].append(pred)
            file_label[fname] = label

        final_preds, final_labels = [], []
        for fname in file_pred_chunks:
            final_labels.append(file_label[fname])
            vulnerable_chunks = sum(1 for pred in file_pred_chunks[fname] if pred == 1)
            if vulnerable_chunks / len(file_pred_chunks[fname]) >= 0.25:
                final_preds.append(1)
            else:
                final_preds.append(0)

        precision, recall, f1, _ = precision_recall_fscore_support(final_labels, final_preds, average='binary')
        acc = accuracy_score(final_labels, final_preds)
        return {
            'accuracy': acc,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        }

    return compute_file_metrics

def tokenize_example(batch, max_length=512):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    filenames_list = []

    for code, label, filename in zip(batch["code"], batch["label"], batch["filename"]):
        tokens = tokenizer(code, return_attention_mask=True, truncation=False)
        input_ids = tokens["input_ids"]
        attention_mask = tokens["attention_mask"]

        for i in range(0, len(input_ids), max_length):
            chunk_ids = input_ids[i:i + max_length]
            chunk_mask = attention_mask[i:i + max_length]

            pad_len = max_length - len(chunk_ids)
            if pad_len > 0:
                chunk_ids += [tokenizer.pad_token_id] * pad_len
                chunk_mask += [0] * pad_len

            input_ids_list.append(chunk_ids)
            attention_mask_list.append(chunk_mask)
            labels_list.append(label)
            filenames_list.append(filename)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "label": labels_list,
        "filename": filenames_list
    }


# Model Finetuning:

In [ ]:
import csv
import os
EPOCHS_LIST = [3, 5]
LEARNING_RATES = [1e-5, 2e-5]
WEIGHT_DECAYS = [0.01]
BATCH_SIZES = [4, 8, 16]
LAYERS_TO_UNFREEZE = [0, 1, 2, 4]

for cwe_id in ALLOWED_CWE_IDS:
    print(f"\n--- Grid Search for {cwe_id} ---")
    samples = collect_files_for_cwe(cwe_id)
    random.seed(SEED)
    random.shuffle(samples)
    raw_dataset = Dataset.from_list(samples)
    tokenized_dataset = raw_dataset.map(tokenize_example, batched=True, remove_columns=["filename", "code"])
    tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label', 'filename'])
    train_test = tokenized_dataset.train_test_split(test_size=0.2, seed=SEED)
    train_dataset = train_test["train"]
    eval_dataset = train_test["test"]
    filenames = eval_dataset["filename"]

    # Prepare log file
    log_path = f"./models/vulberta_{cwe_id}/gridsearch_results.csv"
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epochs", "lr", "weight_decay", "batch_size", "unfrozen_layers", "precision", "recall", "f1", "accuracy", "confusion_matrix"])

    best_f1 = -1
    best_dir = None

    for epochs in EPOCHS_LIST:
        for lr in LEARNING_RATES:
            for wd in WEIGHT_DECAYS:
                for batch_size in BATCH_SIZES:
                    for unfrozen in LAYERS_TO_UNFREEZE:
                        print(f"\nRunning with epochs={epochs}, lr={lr}, wd={wd}, batch_size={batch_size}, unfrozen_layers={unfrozen}")
                        model = AutoModelForSequenceClassification.from_pretrained(vulBERTa, num_labels=2).to(device)

                        for param in model.base_model.parameters():
                            param.requires_grad = False

                        if hasattr(model.base_model, 'encoder'):
                            encoder_layers = model.base_model.encoder.layer
                            if isinstance(encoder_layers, torch.nn.ModuleList):
                                for layer in encoder_layers[-unfrozen:]:
                                    for param in layer.parameters():
                                        param.requires_grad = True

                        for param in model.classifier.parameters():
                            param.requires_grad = True

                        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=wd)
                        num_train_steps = len(train_dataset) * epochs
                        warmup_steps = int(0.1 * num_train_steps)
                        scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, num_train_steps)

                        output_dir = f"./models/vulberta_{cwe_id}/gridsearch/ep{epochs}_lr{lr}_wd{wd}_bs{batch_size}_uf{unfrozen}"
                        training_args = TrainingArguments(
                            output_dir=output_dir,
                            evaluation_strategy="epoch",
                            learning_rate=lr,
                            per_device_train_batch_size=batch_size,
                            per_device_eval_batch_size=batch_size,
                            num_train_epochs=epochs,
                            weight_decay=wd,
                            save_strategy="epoch",
                            load_best_model_at_end=True,
                            metric_for_best_model="eval_loss",
                            remove_unused_columns=False,
                            logging_dir="./logs",
                            logging_strategy="epoch",
                            save_total_limit=1,
                        )

                        trainer = FileAwareTrainer(
                            model=model,
                            args=training_args,
                            train_dataset=train_dataset,
                            eval_dataset=eval_dataset,
                            compute_metrics=compute_file_metrics_builder(filenames),
                            optimizers=(optimizer, scheduler),
                        )

                        trainer.train()
                        trainer.save_model(output_dir + "/final")

                        metrics = trainer.evaluate()
                        precision = metrics["eval_precision"]
                        recall = metrics["eval_recall"]
                        f1 = metrics["eval_f1"]
                        accuracy = metrics["eval_accuracy"]
                        confusion = metrics.get("eval_confusion_matrix", [[-1, -1], [-1, -1]])

                        with open(log_path, "a", newline="") as f:
                            writer = csv.writer(f)
                            writer.writerow([epochs, lr, wd, batch_size, unfrozen, precision, recall, f1, accuracy, confusion])

                        if f1 > best_f1:
                            best_f1 = f1
                            best_dir = output_dir

    if best_dir is not None:
        os.system(f"cp -r {best_dir}/final ./models/vulberta_{cwe_id}/best_model")
        print(f"\nBest model for {cwe_id} saved from: {best_dir} with F1={best_f1:.4f}")



--- Grid Search for CWE-22 ---
320


Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (4354 > 1026). Running this sequence through the model will result in indexing errors



Running with epochs=2, lr=1e-05, wd=0.0, batch_size=4, unfrozen_layers=0


C:\Users\robpi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
C:\Users\robpi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.814200,0.712230,0.458716,0.413043,0.172727,0.243590
2,0.705800,0.731754,0.504587,0.666667,0.036364,0.068966



Running with epochs=2, lr=2e-05, wd=0.0, batch_size=8, unfrozen_layers=0


C:\Users\robpi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
C:\Users\robpi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# Model Evaluation

In [ ]:
def predict_with_chunk_voting(trainer, raw_samples, chunk_size=512, stride=256):
    true_labels = []
    pred_labels = []

    for example in tqdm(raw_samples, desc="Evaluating with chunk voting"):
        label = example["label"]
        true_labels.append(label)

        tokens = tokenizer(example["code"], return_attention_mask=True, truncation=False)
        input_ids = tokens["input_ids"]
        attention_mask = tokens["attention_mask"]

        chunks = []
        for i in range(0, len(input_ids), stride):
            chunk_ids = input_ids[i:i + chunk_size]
            chunk_mask = attention_mask[i:i + chunk_size]

            chunks.append({
                "input_ids": chunk_ids,
                "attention_mask": chunk_mask,
            })

        if not chunks:
            pred_labels.append(0)
            continue

        max_len = max(len(c["input_ids"]) for c in chunks)
        for chunk in chunks:
            pad_len = max_len - len(chunk["input_ids"])
            chunk["input_ids"] += [tokenizer.pad_token_id] * pad_len
            chunk["attention_mask"] += [0] * pad_len

        input_ids = torch.tensor([c["input_ids"] for c in chunks]).to(trainer.model.device)
        attention_mask = torch.tensor([c["attention_mask"] for c in chunks]).to(trainer.model.device)

        with torch.no_grad():
            outputs = trainer.model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()

        file_pred = 1 if (preds.mean() > 0.2) else 0
        pred_labels.append(file_pred)

    return true_labels, pred_labels

true_labels, pred_labels = predict_with_chunk_voting(trainer, samples) 
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary', zero_division=0)
acc = accuracy_score(true_labels, pred_labels)

print(f"Metrics for {cwe_id}:")
print({
    'accuracy': acc,
    'precision': precision,
    'recall': recall,
    'f1': f1,
})

print(f"\nConfusion Matrix for {cwe_id}:")
print(confusion_matrix(true_labels, pred_labels))

Evaluating with chunk voting: 100%|██████████| 320/320 [01:39<00:00,  3.21it/s]

Metrics for CWE-22:
{'accuracy': 0.53125, 'precision': 0.5403225806451613, 'recall': 0.41875, 'f1': 0.47183098591549294}

Confusion Matrix for CWE-22:
[[103  57]
 [ 93  67]]
